In [1]:
import tensorflow as tf
import random
from glob import glob
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import pandas
from sklearn.utils import shuffle
from keras import regularizers
from sklearn.metrics import confusion_matrix

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("shuvoalok/raf-db-dataset")

print("Path to dataset files:", path)

In [3]:
TRAIN_FOLDER_PATH = f"{path}/DATASET/train"
TEST_FOLDER_PATH= f"{path}/DATASET/test"

In [12]:
RAF_LABEL_MAP = {
    "1": "surprise",
    "2": "fear",
    "3": "disgust",
    "4": "happy",
    "5": "sad",
    "6": "angry",
    "7": "neutral",
}

USED_CLASSES = ["angry", "happy", "sad","disgust", "surprise", "neutral"]


def loadImagesFromFolder(
    folderPath,
    labels=None,
    SAMPLE_PERCENT=1.0,
    LIMIT=1700,
    img_size=(48, 48),
    zoom_range=0.3,
    augment=False
):
    subitems = map(Path, glob(folderPath + "/*"))
    subfoldere = list(filter(Path.is_dir, subitems))

    if labels is None:
        labels = []

    X = []
    Y_text = []

    for p in subfoldere:
        folder_name = p.name  # "1", "2",...

        # mapare nr la eticheta text
        if folder_name not in RAF_LABEL_MAP:
            continue
        labelText = RAF_LABEL_MAP[folder_name]

        # excludere clase
        if labelText not in USED_CLASSES:
            continue

        if labelText not in labels:
            labels.append(labelText)

        imagePaths = glob(str(p) + "/*")
        totalImagesCount = len(imagePaths)

        # control special neutral
        if labelText == "neutral":
            local_sample_percent = 0.7 # 70% neutral
            local_limit = 900 # daca nu, 900
        else:
            local_sample_percent = SAMPLE_PERCENT
            local_limit = LIMIT

        random.shuffle(imagePaths)

        sampleImagesCount = int(totalImagesCount * local_sample_percent)
        finalCount = max(1, min(sampleImagesCount, local_limit))

        imagePaths = imagePaths[:finalCount]

        Y_text.extend([labelText] * finalCount)

        for imagePath in imagePaths:
            # grayscale + resize
            img = tf.keras.utils.load_img(
                imagePath,
                color_mode="grayscale",
                target_size=img_size
            )
            img = tf.keras.utils.img_to_array(img)  # (H, W, 1)
            img = img / 255.0

            if augment:
                # zoom random
                if zoom_range > 0:
                    frac = 1.0 - tf.random.uniform([], 0.0, zoom_range)
                    img = tf.image.central_crop(img, central_fraction=frac)
                    img = tf.image.resize(img, img_size)
                # flip random
                img = tf.image.random_flip_left_right(img)

            X.append(img)

    X = np.stack(X, axis=0).astype(np.float32)   # forma - (N, 48, 48, 1)
    Y_text = np.array(Y_text, dtype=str)

    X, Y_text = shuffle(X, Y_text)

    return X, Y_text, labels

In [13]:
X_train, Y_train_text, labels = loadImagesFromFolder(
    TRAIN_FOLDER_PATH, SAMPLE_PERCENT=1, augment=True
)
X_test,  Y_test_text,  _      = loadImagesFromFolder(
    TEST_FOLDER_PATH, labels=labels, SAMPLE_PERCENT=1, augment=False
)


In [14]:
label_encoder = tf.keras.layers.StringLookup(
    vocabulary=labels,
    num_oov_indices=0,
    output_mode="one_hot"
)

Y_train = label_encoder(Y_train_text)
Y_test  = label_encoder(Y_test_text)


In [ ]:
print("Labels order:", labels)
print("Num classes:", len(labels))

Labels order: ['neutral', 'sad', 'disgust', 'surprise', 'happy', 'angry']
Num classes: 6


In [ ]:
print(X_train.shape, X_test.shape)

(4905, 48, 48, 1) (2629, 48, 48, 1)


In [ ]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Conv2D(
        32, (3, 3),
        activation="relu",
        padding="same",
        input_shape=(48, 48, 1)
    ),
    tf.keras.layers.Conv2D(
        64, (3, 3),
        activation="relu",
        padding="same"
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Dropout(0.25),

    tf.keras.layers.Conv2D(
        128, (5, 5),
        activation="relu",
        padding="same"
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Dropout(0.25),

    tf.keras.layers.Conv2D(
        256, (3, 3),
        activation="relu",
        padding="same",
        kernel_regularizer=regularizers.l2(0.001)
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Dropout(0.25),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(256, activation="relu"),
    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Dense(6, activation="softmax"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
history = model.fit(
    X_train, Y_train,
    validation_data=(X_test, Y_test),
    epochs=30,
    batch_size=32,
    shuffle=True
)

In [18]:
loss, acc = model.evaluate(X_test, Y_test)
print("Test accuracy:", acc)

88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.6816 - loss: 1.0895
Test accuracy: 0.6816062927246094


In [ ]:
model.save("modelRAF.h5")